In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)
from pyspark.sql.functions import col, round, current_timestamp

schema = StructType([
    StructField("product_id",  IntegerType(), True),
    StructField("name",        StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("price",       DoubleType(),  True),
    StructField("stock",       IntegerType(), True),
])

data = [
    (1,  "Laptop",      "Electronics", 999.99,  50),
    (2,  "Phone",       "Electronics", 699.99, 120),
    (3,  "Desk Chair",  "Furniture",   249.99,  30),
    (4,  "Notebook",    "Stationery",    4.99, 500),
    (5,  "Headphones",  "Electronics", 149.99,  75),
    (6,  "Desk Lamp",   "Furniture",    39.99,  90),
    (7,  "Pen Set",     "Stationery",    9.99, 300),
    (8,  "Monitor",     "Electronics", 399.99,  40),
]

df = spark.createDataFrame(data, schema)
df.show()

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("products")

print("Method 1: saveAsTable — registers in metastore by name")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS products_sql (
    product_id  INT,
    name        STRING,
    category    STRING,
    price       DOUBLE,
    stock       INT
) USING DELTA;

INSERT INTO products_sql VALUES
    (1, 'Laptop',     'Electronics', 999.99,  50),
    (2, 'Phone',      'Electronics', 699.99, 120),
    (3, 'Desk Chair', 'Furniture',   249.99,  30);

SELECT * FROM products_sql;

In [0]:
spark.sql("""
    create table if not exists products_spark
    using delta
    as select * from products
""")

spark.read.table("products_spark").show()
print("Method 3: create as select - creates and populates in one step")

In [0]:
%sql
alter table products 
add columns(
  discount_pct double,
  last_updated string
);

select * from products limit 3;

In [0]:
from pyspark.sql.functions import lit, current_date

df_updated = spark.read.table("products") \
    .withColumn("discount_pct", lit(0.10)) \
    .withColumn("last_updated", current_date().cast("string"))

df_updated.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("products")

spark.read.table("products").show()

In [0]:
%sql 
update products
set price = round(price * 0.9, 2),
    last_updated = current_date()
where category = 'Electronics';

select * from products
where category = 'Electronics'

In [0]:
%sql
delete from products
where stock = 0;

select count(*) as remaining_products from products;

In [0]:
%sql
describe history products

In [0]:
%sql
optimize products;

In [0]:
%sql 
vacuum products retain 168 hours;

In [0]:
%sql
alter table products 
add columns(
  value_in_stock double
);

update products
set value_in_stock = price * stock,
    price = price * 1.15,
    last_updated = current_date();




In [0]:
%sql
describe history products

In [0]:

%sql
select * from products version as of 0

In [0]:
%sql
SELECT
    current.name,
    current.category,
    original.price AS original_price,
    current.price  AS current_price,
    ROUND(current.price - original.price, 2) AS price_change
FROM products current
JOIN (SELECT name, price FROM products VERSION AS OF 0) original
    ON current.name = original.name
WHERE current.category = 'Furniture'